In [4]:
pip install pylint

Note: you may need to restart the kernel to use updated packages.


In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cmath as m
import math as ma
import plotly.express as px
import plotly.graph_objects as go
from scipy import special as sp
from scipy.special import erf
from variables import *

ModuleNotFoundError: No module named 'variables'

In [ ]:
def read_complex_matrix(filename):
    matrix = []
    with open(filename, 'r') as file:
        for line in file:
            values = line.strip('[] \n').split(',')
            row = [complex(x.strip()) for x in values]
            matrix.append(row)
    return np.array(matrix)
matr = read_complex_matrix('/content/ref_matrix.txt')

In [ ]:
def transform_array(input_array, N, function):
    result = np.column_stack((input_array[:, 0], input_array[:, 1] - input_array[:, 2]))[:, :2].tolist()  #(ro, d)
    # Добавил промежуточные точки
    for i in range(1, len(input_array)):
        d2, ro2, rough2 = input_array[i]
        ro1 = input_array[i-1][1]
        insert_index = len(result) - (len(input_array) - i)
        for j in range(0, N+1):
            new_d = (rough2) / N
            new_ro =  ((ma.erf(-4*j/N+2)+1)/2)*(ro2-ro1)+ro1
            result.insert(insert_index, [new_d, new_ro])
    return np.array(result, dtype=complex)

In [ ]:
def plot_density_profile(matrix):
    matrix = np.array(matrix)
    depth_bins = np.cumsum(np.append(0, (1e+10)*matrix[:, 0].real))
    depth_bins = np.append(depth_bins, depth_bins[-1] + (1e+10)*matrix[-1, 0].real)
    U = (1e-14)*matrix[:, 1].real
    U = np.append(U, U[-1])
    x_fill = np.repeat(depth_bins[:-1], 2)
    y_fill = np.repeat(U, 2)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_fill,
        y=y_fill,
        fill='tozeroy',
        mode='none',
        fillcolor='rgba(0,0,255,0.15)',
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=depth_bins[:-1],
        y=U,
        mode='lines',
        line=dict(color='blue', shape='hv'),
        showlegend=False
    ))
    fig.add_hline(y=0, line=dict(color='black', width=2))
    fig.update_layout(
        xaxis_title='Глубина, Å',
        yaxis_title='Плотность длины рассеяния, 10^-6 Å^2',
        xaxis=dict(range=[0, depth_bins[-1]]),
        showlegend=False,
        plot_bgcolor='white'
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

    return fig

In [ ]:
matrix = transform_array(matr, rough_res, lambda z: z)
fig = plot_density_profile(transform_array(matr, rough_res, lambda z: z))
fig.show()

In [ ]:
#объявляем переменные
ro = np.array(transform_array(matr, rough_res, lambda z: z))[:, 1]
d = np.array(transform_array(matr, rough_res, lambda z: z))[:,0]
q = np.empty((len(ro), Ndots), dtype = complex)
dm = np.empty((len(ro), Ndots, 2,2), dtype = complex)
dmi = np.empty((len(ro), Ndots, 2,2), dtype = complex)
pm = np.empty((len(ro), Ndots, 2,2), dtype = complex)
M = np.empty((Ndots, 2,2), dtype = complex)
dk0 = m.sqrt(m.sqrt(np.mean(ro**2))/Ndots)
Pe = np.array([[1, 0], [0, 1]])
i_indices = np.arange(Ndots) + 0.001
j_indices = np.arange(len(ro))
j_grid, i_grid = np.meshgrid(j_indices, i_indices, indexing='ij')
q = np.sqrt((i_grid * dk0)**2 - 12.56637 * ro[j_grid])
dm[j_grid, i_grid.astype(int), :, :] = np.array([[np.ones((len(ro), Ndots)), np.ones((len(ro), Ndots))], [q, -q]]).transpose(2, 3, 0, 1)
phase = q * d[j_grid]
exp_neg = np.exp(1j * (-1 * phase))
exp_pos = np.exp(1j * phase)
zeros = np.zeros_like(q)
pm[j_grid, i_grid.astype(int), :, :] = np.array([
    [exp_neg, zeros],
    [zeros, exp_pos]
]).transpose(2, 3, 0, 1)

In [ ]:
#считаем матрицы
dmi = np.linalg.inv(dm.reshape(-1, 2, 2)).reshape(len(ro), Ndots, 2, 2)
if dm.shape[0] == 2:
  M = (
     dmi[0] @ dm[1]
  )
if dm.shape[0] == 3:
  M = (
     dmi[0] @ dm[1] @ pm[1] @ dmi[1] @ dm[2]
  )
if dm.shape[0] > 3:
  for j in range(1, dm.shape[0]-1):
    Pe = (
      dm[j] @ pm[j] @ dmi[j] @ Pe
    )
  M = dmi[0] @ Pe @ dm[-1]
r = M[:,1,0]/M[:,0,0]

In [ ]:
r_real = []
r_img = []
r_abs = []
q0_a = []
for i in range (0, Ndots):
  r_real.append(r[i].real)
  r_img.append(r[i].imag)
  r_abs.append(m.polar(r[i])[0])
  q0_a.append((i+0.9)*dk0.real*1e-10)

In [ ]:
fig = px.scatter(x=r_img, y=r_real, color=q0_a,
                 color_continuous_scale='hot',
                 title='complex reflection ratio')
fig.update_layout(
    xaxis=dict(scaleanchor="y"),  # Привязка оси X к оси Y
    yaxis=dict(constrain='domain')  # Ограничение оси Y
)
fig.update_traces(marker=dict(size=2))
fig.show()

In [ ]:
fig = px.scatter(x=q0_a, y=r_abs,
                 title='График зависимости коэффициента отражения от волнового вектора',
                 labels={'x': 'Значения исходного волнового вектора, Å^-1',
                         'y': 'Значения коэффициента отражения'})
fig.update_yaxes(type='log')
fig.update_traces(marker=dict(size=1.5))
fig.show()